# Bound row sampling before feature selection

Compare a seeded random sample, a Gower cluster sample and isolation
sampling on a 1,200-row synthetic training fixture. The target travels with
the training rows and is removed from the predictors before selection.
Because Gower clusters this entire table, the response participates in
grouping. Use a predictors-only sampling design when target values cannot
be used to pick rows.

In [1]:
import numpy as np
import pandas as pd

from arfs.feature_selection import MinRedundancyMaxRelevance
from arfs.sampling import sample

rng = np.random.default_rng(42)
data = pd.DataFrame(rng.normal(size=(1_200, 4)), columns=["x1", "x2", "x3", "x4"])
data["group"] = pd.Categorical(rng.choice(["a", "b", "c"], size=len(data)))
data["response"] = 4 * data["x1"] - 2 * data["x2"] + rng.normal(size=len(data))

## Cap the rows passed to Gower

Exact Gower clustering stores a condensed float64 distance vector.
For N input rows, this takes 4 × N × (N − 1) bytes before clustering
and temporary arrays. At 50,000 rows that is about 10 GB, and at 100,000
about 40 GB. The requested output size does not lower this cost.
The 600-row presample below takes about 1.4 MB for distances.

Gower returns synthetic cluster representatives with column means or modes.
Isolation uses an unweighted distribution check on anomaly scores and
returns original rows. If it finds no acceptable proper subset, it
returns all rows it received. The [sampling limits](../workflows.rst#sampling-limits) section describes
observation-weight behavior and memory in more detail.

In [2]:
presample = data.sample(n=min(len(data), 600), random_state=42)
gower = sample(presample, n=120, method="gower")

## Compare selections at this scale

Request two predictors with MRMR for each training table. Compare the
selected names and row counts. To check whether sampling is suitable for
your task, compare across draws and evaluate downstream models on the same
untouched rows.

In [3]:
samples = {
    "Full training table": data,
    "Random rows": data.sample(n=120, random_state=42),
    "Gower representatives": gower,
    "Isolation rows": sample(presample, method="isoforest", random_state=42),
}

rows = []
for name, table in samples.items():
    X = table.drop(columns="response")
    y = table["response"]
    selector = MinRedundancyMaxRelevance(
        n_features_to_select=2, task="regression", n_jobs=1, show_progress=False
    ).fit(X, y)
    rows.append((name, len(table), selector.selected_features_))

pd.DataFrame(rows, columns=["Training table", "Rows", "Selected columns"])

,Training table,Rows,Selected columns
0,Full training table,1200,"[x1, x3]"
1,Random rows,120,"[x1, group]"
2,Gower representatives,120,"[x1, x2]"
3,Isolation rows,30,"[x1, x2]"
